In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys

project_path = "/content/drive/MyDrive/Colab Notebooks"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import src

importlib.invalidate_caches()

import src.weather as weather

import src.crops as crops

import src.irrigation as irrigation

import src.fao56 as fao

import src.advisor as advisor

importlib.reload(irrigation)

importlib.reload(crops)

82.75438254773033
0.05503166439424067
2.338281270927446
2.921959783327874


<module 'src.crops' from '/content/drive/MyDrive/Colab Notebooks/src/crops.py'>

## load the dataset

In [ ]:
df = pd.read_csv("/content/kiambu_weather_2016_2025.csv")

df.head()

,Date,Temperature,Max_Temperature,Min_Temperature,Relative_Humidity,Rainfall,Wind_Speed,Solar_Radiation
0,2016-01-01,19.94,26.62,14.85,72.57,0.01,2.70,26.93
1,2016-01-02,18.89,26.15,13.02,73.27,0.49,2.97,25.65
2,2016-01-03,18.77,25.95,13.02,78.50,5.70,3.29,24.74
3,2016-01-04,18.80,24.16,15.21,83.54,11.28,4.17,16.70
4,2016-01-05,18.58,24.22,15.05,82.98,7.03,4.02,24.17


convert the dates

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfYear"] = df["Date"].dt.dayofyear
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

In [ ]:
df.head(15)

,Date,Temperature,Max_Temperature,Min_Temperature,Relative_Humidity,Rainfall,Wind_Speed,Solar_Radiation,Year,Month,Day,DayOfYear,Week
0,2016-01-01,19.94,26.62,14.85,72.57,0.01,2.70,26.93,2016,1,1,1,53
1,2016-01-02,18.89,26.15,13.02,73.27,0.49,2.97,25.65,2016,1,2,2,53
2,2016-01-03,18.77,25.95,13.02,78.50,5.70,3.29,24.74,2016,1,3,3,53
3,2016-01-04,18.80,24.16,15.21,83.54,11.28,4.17,16.70,2016,1,4,4,1
4,2016-01-05,18.58,24.22,15.05,82.98,7.03,4.02,24.17,2016,1,5,5,1
5,2016-01-06,19.10,25.06,14.47,79.57,0.79,3.30,24.46,2016,1,6,6,1
6,2016-01-07,18.76,25.20,14.26,78.41,0.10,3.03,24.49,2016,1,7,7,1
7,2016-01-08,18.87,25.80,13.10,74.91,0.02,2.79,25.45,2016,1,8,8,1
8,2016-01-09,18.94,25.71,13.17,74.38,0.01,2.72,23.15,2016,1,9,9,1
9,2016-01-10,19.24,24.88,15.34,73.52,0.04,2.19,15.13,2016,1,10,10,1


# temperature range

In [ ]:
df["Temp_Range"] = (
    df["Max_Temperature"]
    - df["Min_Temperature"]
)

# rolling rainfal

In [ ]:
df["Rain_3Day"] = (
    df["Rainfall"]
    .rolling(3)
    .sum()
)

df["Rain_7Day"] = (
    df["Rainfall"]
    .rolling(7)
    .sum()
)

# Rolling temperature

In [ ]:
df["Temp_3Day"] = (
    df["Temperature"]
    .rolling(3)
    .mean()
)

df["Temp_7Day"] = (
    df["Temperature"]
    .rolling(7)
    .mean()
)

In [ ]:
def calculate_daily_et0_from_radiation(
    Tmax,
    Tmin,
    RH,
    wind_speed,
    elevation,
    latitude_deg,
    day_of_year,
    solar_radiation,
):
    """
    Calculate ET0 using measured solar radiation.
    """

In [ ]:

import importlib
import src.fao56 as fao

importlib.reload(fao)


82.75438254773033
0.05503166439424067
2.338281270927446
2.921959783327874


<module 'src.fao56' from '/content/drive/MyDrive/Colab Notebooks/src/fao56.py'>

## compute the ETo for the entire dataset

In [ ]:
LATITUDE = -1.286        # Kiambu (adjust if needed)
ELEVATION = 1700         # metres
WIND_HEIGHT = 2          # metres

In [ ]:
def compute_et0(row):

    return fao.calculate_daily_et0_from_radiation(

        Tmax=row["Max_Temperature"],
        Tmin=row["Min_Temperature"],
        RH=row["Relative_Humidity"],
        wind_speed=row["Wind_Speed"],
        wind_height=WIND_HEIGHT,
        elevation=ELEVATION,
        latitude_deg=LATITUDE,
        day_of_year=row["DayOfYear"],
        solar_radiation=row["Solar_Radiation"],
    )

In [ ]:
df["ET0"] = df.apply(
    compute_et0,
    axis=1
)

In [ ]:
df[["Date", "ET0"]].head()

,Date,ET0
0,2016-01-01,5.025956
1,2016-01-02,4.732370
2,2016-01-03,4.407417
3,2016-01-04,3.230699
4,2016-01-05,4.089227


In [3]:
import src.weather as weather

print(weather.sample_weather())

{'Tmax': 28.0, 'Tmin': 18.0, 'RH': 70, 'wind_speed': 2.0, 'wind_height': 2.0, 'rainfall': 2.4, 'sunshine_hours': 8.5, 'latitude_deg': -1.246, 'longitude_deg': 36.662, 'elevation': 1800, 'day_of_year': 100}


In [4]:
from datetime import datetime

import src.crop_calendar as crop

planting = datetime(2025,3,1)

today = datetime(2025,4,15)

print(crop.crop_age(today, planting))

print(crop.get_growth_stage(today, planting))

print(crop.get_kc(today, planting))

print(crop.is_crop_active(today, planting))

print(crop.is_stage(today, planting, "Development"))


45
Development
0.9
True
True
